In [ ]:
%pip install --upgrade datasets
%pip install transformers accelerate torch sentencepiece wandb

In [1]:
# Cell 2: Imports (fixed)
import torch
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer
from secret_key import get_secret_key
from huggingface_hub import login

print("done")

c:\Users\diata\Documents\Bagas\Tugas\UNPAD\Semester 4\AI\Kelas\AI Question Generator\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


done


In [2]:
dataset = load_dataset("rajpurkar/squad")
# Filter to specified number of samples
train_dataset = dataset['train'].select(range(min(10000, len(dataset['train']))))
val_dataset = dataset['validation'].select(range(min(1000, len(dataset['validation']))))
print("done")

done


In [3]:
MODEL_PRETRAINED = "openai-community/gpt2"
MODEL_FINETUNED_PATH = "gpt2-finetuned"
TRAIN_SAMPLES = 10000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

Device: cpu


In [4]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PRETRAINED)
tokenizer.pad_token = tokenizer.eos_token
print("done")

done


In [ ]:
def preprocess_function(examples):
    inputs = [f"Context: {ctx} Question:" for ctx in examples['context']]
    targets = examples['question']
    
    model_inputs = tokenizer(inputs, max_length=400, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=100, truncation=True, padding="max_length")
    
    combined_inputs = []
    combined_labels = []
    
    for i in range(len(inputs)):
        # Combine input + target
        full_text = inputs[i] + " " + targets[i] + tokenizer.eos_token
        tokenized = tokenizer(full_text, max_length=512, truncation=True, padding="max_length")
        
        combined_inputs.append(tokenized['input_ids'])
        combined_labels.append(tokenized['input_ids'].copy())
        
        input_length = len(tokenizer(inputs[i])['input_ids'])
        combined_labels[-1][:input_length] = [-100] * input_length
    
    return {
        'input_ids': combined_inputs,
        'attention_mask': [tokenizer(f"Context: {ctx} Question: {q}{tokenizer.eos_token}", 
                                   max_length=512, truncation=True, padding="max_length")['attention_mask'] 
                          for ctx, q in zip(examples['context'], targets)],
        'labels': combined_labels
    }

# Apply preprocessing
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
print("done")

Map: 100%|██████████| 1000/1000 [00:03<00:00, 270.29 examples/s]

done


In [ ]:
model = GPT2LMHeadModel.from_pretrained(MODEL_PRETRAINED)
model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir=MODEL_FINETUNED_PATH,
    num_train_epochs=3,
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    save_steps=500,
    evaluation_strategy="steps",
    eval_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model()